### Import packages

In [ ]:
from pathlib import Path
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import re
import math

### Define paths

In [12]:
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"

GENRE_PATHS = {
    "romance": DATA_DIR / "romance_stories_cleaned.jsonl",
    "literary_fiction": DATA_DIR / "lit_fiction_stories_cleaned.jsonl",
    "sci_fi": DATA_DIR / "sci_fi_stories_cleaned.jsonl",
}

In [ ]:
# chose genre
genre = "sci_fi"

### Load model

In [14]:
# Load model and tokenizer
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1859.12it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Define functions

In [15]:
def load_jsonl(path, limit=100):
    stories = []
    
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if i >= limit:
                break
            stories.append(json.loads(line))
    
    return stories

In [16]:
def split_sentences(text):
    return re.split(r'(?<=[.!?]) +', text)

In [17]:
def get_toxicity_scores(texts, batch_size=16):
    all_scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        with torch.no_grad():
            logits = model(**inputs).logits
            
            # Convert logits to probabilities
            probs = torch.sigmoid(logits)
            # take probability of the toxic class (index 1)
            # Only take "toxic" (label 0)
            toxic_probs = probs[:, 0].tolist()
            all_scores.extend(toxic_probs)  

    return all_scores

In [18]:
def compute_story_toxicity(stories):
    for story_dict in stories:
        sentences = split_sentences(story_dict["story"])

        if not sentences:
            story_dict["average_toxicity"] = 0
            continue

        scores = get_toxicity_scores(sentences)
        story_dict["average_toxicity"] = sum(scores) / len(scores)

    return stories

In [19]:
def compute_corpus_stats(stories):
    values = [s["average_toxicity"] for s in stories]
    n = len(values)
    
    if n == 0:
        raise ValueError("Cannot compute statistics on an empty list.")
    
    mean = sum(values) / n
    
    # Sample standard deviation (n - 1)
    variance = sum((x - mean) ** 2 for x in values) / (n - 1) if n > 1 else 0.0
    std = math.sqrt(variance)
    
    return mean, std

### Compute toxicity

In [20]:
stories = load_jsonl(GENRE_PATHS[genre])
stories = compute_story_toxicity(stories)

avg_toxicity = compute_corpus_stats(stories)

print(avg_toxicity)

(0.007873713317247873, 0.005318378767570401)
